# 09 — Eval teacher base vs warmup on Connections (JSON-only)

Forza output **SOLO JSON** per i gruppi (nessun ragionamento). Include:
- prompt senza placeholder
- estrazione robusta del JSON
- validazione strict (solo parole date, 4x4, copertura 16)
- metriche order-invariant


In [1]:
from pathlib import Path

BASE_MODEL_ID = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
WARM_MODEL_PATH = Path("../checkpoints/teacher_warmup_atomic_swow/final").resolve()

CONNECTIONS_JSONL = Path("../data/processed/hf_train_dedup.jsonl").resolve()

N_SAMPLES = 10
SEED = 1234

#MAX_NEW_TOKENS = 500
DO_SAMPLE = True
TEMPERATURE = 0.6
TOP_P = 0.95

print("Warm model path exists:", WARM_MODEL_PATH.exists())
print("Dataset exists:", CONNECTIONS_JSONL.exists())


Warm model path exists: True
Dataset exists: True


## 1) Load JSONL

In [2]:
import json, random
from typing import Any, Dict, List

def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def sample_rows(rows, n, seed):
    if n is None or n <= 0 or n >= len(rows):
        return rows
    rng = random.Random(seed)
    idx = list(range(len(rows)))
    rng.shuffle(idx)
    return [rows[i] for i in idx[:n]]

rows = read_jsonl(CONNECTIONS_JSONL)
rows = sample_rows(rows, N_SAMPLES, SEED)
print("Loaded:", len(rows))
print("Example keys:", rows[0].keys())


Loaded: 10
Example keys: dict_keys(['puzzle_id', 'date', 'words', 'answers', 'metadata'])


## 2) Prompt JSON-only (no system prompt)

In [3]:
from typing import List, Dict

def build_messages(words16: List[str]) -> List[Dict[str, str]]:
    user = (
        "Solve NYT Connections, i'm playing a game in order to win money for college, so is MANDATORY to respect the following rules.\n"
        "You are given exactly 16 words.\n"
        "\n\nRules:\n"
        "- Exactly 4 groups.\n"
        "- Each group has exactly 4 words.\n"
        "- Use ONLY the provided words, uppercase exactly as given.\n"
        "- Do NOT use placeholders like W1/W2.\n\n"
        "- DO NOT GENERATE ANY OTHER TEXT WHEN GIVING THE ANSWER, JUST THE 4 GROUPS OF 4 WORDS"
        "Words: " + ", ".join(words16)
    )
    return [{"role": "user", "content": user}]


## 3) Gold + parsing/validation

In [4]:
import re, itertools, json
from typing import Any, Dict, List, Optional, Set

def normalize_word(w: str) -> str:
    return " ".join(str(w).strip().split())

def extract_gold_groups(example: Dict[str, Any]) -> List[Set[str]]:
    answers = example.get("answers")
    if not answers:
        raise KeyError("Missing 'answers' field.")
    groups = []
    for g in answers:
        ws = g.get("words") or g.get("Words") or g.get("items") or g.get("group")
        if ws is None:
            raise KeyError("Each answer group must contain 'words' list.")
        groups.append(set(normalize_word(x).upper() for x in ws))
    if len(groups) != 4:
        raise ValueError(f"Expected 4 gold groups, got {len(groups)}")
    return groups

def extract_first_json_object(text: str) -> Optional[str]:
    if "<｜Assistant｜>" in text:
        text = text.split("<｜Assistant｜>", 1)[1]
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.S)
    m = re.search(r"\{.*\}", text, flags=re.S)
    if not m:
        return None
    return m.group(0).strip()

def parse_pred_groups(text: str, words16: List[str]) -> Optional[List[Set[str]]]:
    blob = extract_first_json_object(text)
    if blob is None:
        return None
    try:
        obj = json.loads(blob)
    except Exception:
        return None
    groups = obj.get("groups")
    if not (isinstance(groups, list) and len(groups) == 4):
        return None
    out: List[Set[str]] = []
    for g in groups:
        if not (isinstance(g, list) and len(g) == 4):
            return None
        s = set(normalize_word(x).upper() for x in g)
        if len(s) != 4:
            return None
        out.append(s)

    wordset = set(w.upper() for w in words16)
    flat = set().union(*out)
    if not flat.issubset(wordset):
        return None
    if len(flat) != 16:
        return None
    return out

def best_match_count(pred: List[Set[str]], gold: List[Set[str]]) -> int:
    best = 0
    for perm_gold in itertools.permutations(gold, 4):
        for perm_pred in itertools.permutations(pred, 4):
            score = sum(1 for i in range(4) if perm_pred[i] == perm_gold[i])
            best = max(best, score)
            if best == 4:
                return 4
    return best


## 4) Load models (4-bit) + generate

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

def load_4bit_model(model_id_or_path: str):
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    tok = AutoTokenizer.from_pretrained(model_id_or_path, use_fast=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_id_or_path,
        quantization_config=bnb,
        device_map="auto",
        torch_dtype=torch.float16,
    )
    model.eval()
    return tok, model

@torch.inference_mode()
def generate_json(tok, model, messages):
    if hasattr(tok, "apply_chat_template"):
        prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tok(prompt, return_tensors="pt").to(model.device)
    else:
        prompt = ""
        for m in messages:
            prompt += f"{m['role'].upper()}: {m['content']}\n"
        prompt += "ASSISTANT:"
        inputs = tok(prompt, return_tensors="pt").to(model.device)

    gen_kwargs = dict(pad_token_id=tok.eos_token_id)
        #gen_kwargs = dict(max_new_tokens=MAX_NEW_TOKENS, pad_token_id=tok.eos_token_id)

    if DO_SAMPLE:
        gen_kwargs.update(dict(do_sample=True, temperature=TEMPERATURE, top_p=TOP_P))
    else:
        gen_kwargs.update(dict(do_sample=False))

    out = model.generate(**inputs, **gen_kwargs)
    txt = tok.decode(out[0], skip_special_tokens=True)
    if txt.startswith(prompt):
        txt = txt[len(prompt):].strip()
    return txt


c:\Users\cola0\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 5) Evaluation

In [6]:
from tqdm import tqdm
import pandas as pd

tok_base, model_base = load_4bit_model(BASE_MODEL_ID)
#tok_warm, model_warm = load_4bit_model(str(WARM_MODEL_PATH))

def eval_model(tok, model, name: str, debug_max: int = 3):
    n = 0
    parsed = 0
    puzzle_exact_sum = 0
    groups_correct_sum = 0
    dbg = 0

    for ex in tqdm(rows, desc=name):
        words16 = ex["words"]
        gold = extract_gold_groups(ex)

        raw = generate_json(tok, model, build_messages(words16))
        pred = parse_pred_groups(raw, words16)

        n += 1
        if pred is None:
            if dbg < debug_max:
                print("\n" + "="*90)
                print(f"[DEBUG] {name} parse failed #{dbg+1}")
                print("RAW OUTPUT (repr, first 800):", repr(raw))
                print("WORDS:", words16)
                print("="*90)
                dbg += 1
            continue

        parsed += 1
        best = best_match_count(pred, gold)
        groups_correct_sum += best
        if best == 4:
            puzzle_exact_sum += 1

    return {
        "model": name,
        "n": n,
        "parsed_rate": parsed / n if n else 0.0,
        "puzzle_exact": puzzle_exact_sum / n if n else 0.0,
        "group_match_rate": groups_correct_sum / (4*n) if n else 0.0,
        "failed_parses": n - parsed,
    }

res_base = eval_model(tok_base, model_base, "teacher_base")
#res_warm = eval_model(tok_warm, model_warm, "teacher_warmup")

#pd.DataFrame([res_base, res_warm])
res_base


`torch_dtype` is deprecated! Use `dtype` instead!
teacher_base:  10%|█         | 1/10 [00:01<00:10,  1.11s/it]


[DEBUG] teacher_base parse failed #1
RAW OUTPUT (repr, first 800): "<｜User｜>Solve NYT Connections, i'm playing a game in order to win money for college, so is MANDATORY to respect the following rules.\nYou are given exactly 16 words.\n\n\nRules:\n- Exactly 4 groups.\n- Each group has exactly 4 words.\n- Use ONLY the provided words, uppercase exactly as given.\n- Do NOT use placeholders like W1/W2.\n\n- DO NOT GENERATE ANY OTHER TEXT WHEN GIVING THE ANSWER, JUST THE 4 GROUPS OF 4 WORDSWords: JAM, PACK, RAM, STUFF, COOK, DISHWASHER, HOST, SERVER, MICROWAVE, RADIO, VISIBLE, X-RAY, BRIDLE, BYTE, COMEDIAN, DRILL<｜Assistant｜><think>\nAlright, so I need to solve this New York Times Connections game. The goal is to win money"
WORDS: ['JAM', 'PACK', 'RAM', 'STUFF', 'COOK', 'DISHWASHER', 'HOST', 'SERVER', 'MICROWAVE', 'RADIO', 'VISIBLE', 'X-RAY', 'BRIDLE', 'BYTE', 'COMEDIAN', 'DRILL']


teacher_base:  20%|██        | 2/10 [00:01<00:07,  1.08it/s]


[DEBUG] teacher_base parse failed #2
RAW OUTPUT (repr, first 800): "<｜User｜>Solve NYT Connections, i'm playing a game in order to win money for college, so is MANDATORY to respect the following rules.\nYou are given exactly 16 words.\n\n\nRules:\n- Exactly 4 groups.\n- Each group has exactly 4 words.\n- Use ONLY the provided words, uppercase exactly as given.\n- Do NOT use placeholders like W1/W2.\n\n- DO NOT GENERATE ANY OTHER TEXT WHEN GIVING THE ANSWER, JUST THE 4 GROUPS OF 4 WORDSWords: APPROVAL, BLESSING, CONSENT, SUPPORT, BAGEL, LIFESAVER, TIRE, WREATH, HOOK, SHANK, SLICE, WHIFF, LOAF, SLIP, SNEAK, WADE<｜Assistant｜><think>\nOkay, so I need to solve this New York Times Connections game. The goal is to win money"
WORDS: ['APPROVAL', 'BLESSING', 'CONSENT', 'SUPPORT', 'BAGEL', 'LIFESAVER', 'TIRE', 'WREATH', 'HOOK', 'SHANK', 'SLICE', 'WHIFF', 'LOAF', 'SLIP', 'SNEAK', 'WADE']


teacher_base:  30%|███       | 3/10 [00:02<00:06,  1.16it/s]


[DEBUG] teacher_base parse failed #3
RAW OUTPUT (repr, first 800): "<｜User｜>Solve NYT Connections, i'm playing a game in order to win money for college, so is MANDATORY to respect the following rules.\nYou are given exactly 16 words.\n\n\nRules:\n- Exactly 4 groups.\n- Each group has exactly 4 words.\n- Use ONLY the provided words, uppercase exactly as given.\n- Do NOT use placeholders like W1/W2.\n\n- DO NOT GENERATE ANY OTHER TEXT WHEN GIVING THE ANSWER, JUST THE 4 GROUPS OF 4 WORDSWords: KEEP, PRESERVE, SAVE, STORE, BUFF, FILE, GRIND, SAND, FAVORITE, PARLAY, SPREAD, UNDER, BUTTER, CHICKEN, LADY, STICKY<｜Assistant｜><think>\nOkay, so I'm trying to solve this New York Times Connections game. I have 16 words"
WORDS: ['KEEP', 'PRESERVE', 'SAVE', 'STORE', 'BUFF', 'FILE', 'GRIND', 'SAND', 'FAVORITE', 'PARLAY', 'SPREAD', 'UNDER', 'BUTTER', 'CHICKEN', 'LADY', 'STICKY']


teacher_base: 100%|██████████| 10/10 [00:08<00:00,  1.21it/s]


{'model': 'teacher_base',
 'n': 10,
 'parsed_rate': 0.0,
 'puzzle_exact': 0.0,
 'group_match_rate': 0.0,
 'failed_parses': 10}

### Se `parsed_rate` è bassa
1) prova `DO_SAMPLE=True`, `TEMPERATURE=0.6`, `TOP_P=0.95`
2) aumenta MAX_NEW_TOKENS a 192
